In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

In [2]:
def test_accuracy(df, dataset, model, windows=["w_00_40", "w_05_45"], model_name=None, mode=None):
    df = df[(df["dataset"] == dataset) & (df["model"] == model)]

    if mode is not None:
        df = df[df["mode"] == mode]

    if "session" in list(df.columns):
        df = df.groupby(["mode", "dataset", "subject", "model", "epoch_window"])["accuracy"].mean().reset_index()

    a = df[df["epoch_window"] == windows[0]].sort_values("subject")["accuracy"].to_numpy()
    b = df[df["epoch_window"] == windows[1]].sort_values("subject")["accuracy"].to_numpy()

    stats = ttest_rel(a=a, b=b, nan_policy="raise", alternative="two-sided")

    if mode is None:
        print(f"{dataset}, {model_name}: {stats.pvalue}")
    else:
        print(f"{dataset}, {model_name}, {mode}: {stats.pvalue}")

    return pd.DataFrame({"dataset": [dataset], "model": [model_name], "mode": [mode], "pvalue": [stats.pvalue]})



In [3]:
df_list = []

# Deep Learning

In [4]:
df = pd.read_csv(Path("../") / "results" / "classification_results.csv")
df

,subject,dataset,model,epoch_window,accuracy
0,1,Lee2019_MI,Deep4Net,w_00_30,0.83000
1,1,Lee2019_MI,Deep4Net,w_05_35,0.77000
2,2,Lee2019_MI,Deep4Net,w_00_30,0.79000
3,2,Lee2019_MI,Deep4Net,w_05_35,0.74000
4,3,Lee2019_MI,Deep4Net,w_00_30,0.84000
...,...,...,...,...,...
721,85,Dreyer2023_wo_p,Deep4Net,w_00_40,0.81875
722,86,Dreyer2023_wo_p,Deep4Net,w_05_45,0.69375
723,86,Dreyer2023_wo_p,Deep4Net,w_00_40,0.86875
724,87,Dreyer2023_wo_p,Deep4Net,w_05_45,0.89375


In [5]:
df_list.append(
    test_accuracy(df, "Dreyer2023", windows=["w_00_40", "w_05_45"], model="Deep4Net", model_name="DeepConvNet"))
df_list.append(test_accuracy(df, "Dreyer2023", windows=["w_00_40", "w_05_45"], model="REVE", model_name="REVE"))
df_list.append(
    test_accuracy(df, "Lee2019_MI", windows=["w_00_30", "w_05_35"], model="Deep4Net", model_name="DeepConvNet"))
df_list.append(test_accuracy(df, "Lee2019_MI", windows=["w_00_30", "w_05_35"], model="REVE", model_name="REVE"))
test_accuracy(df, "Dreyer2023_wo_p", windows=["w_00_40", "w_05_45"], model="Deep4Net", model_name="DeepConvNet")


Dreyer2023, DeepConvNet: 8.693759346417105e-18
Dreyer2023, REVE: 2.49722666675134e-12
Lee2019_MI, DeepConvNet: 1.3169748132231866e-07
Lee2019_MI, REVE: 9.646089451560376e-07
Dreyer2023_wo_p, DeepConvNet: 7.585452462142225e-15


,dataset,model,mode,pvalue
0,Dreyer2023_wo_p,DeepConvNet,None,7.585452e-15


# CSP+LDA

In [6]:
df_csp = pd.read_csv(Path("../") / "results" / "classification_results_csp_lda.csv")
df_csp

,mode,dataset,subject,session,model,epoch_window,accuracy
0,within_user,Lee2019_MI,1,1,CSP_LDA,w_05_35,0.52000
1,within_user,Lee2019_MI,1,2,CSP_LDA,w_05_35,0.46000
2,within_user,Dreyer2023,1,1,CSP_LDA,w_00_40,0.76875
3,within_user,Dreyer2023,1,1,CSP_LDA,w_05_45,0.83750
4,within_user,Lee2019_MI,1,1,CSP_LDA,w_00_30,0.56000
...,...,...,...,...,...,...,...
767,cross_user,Dreyer2023,86,1,CSP_LDA,w_05_45,0.51250
768,within_user,Dreyer2023,87,1,CSP_LDA,w_05_45,0.84375
769,cross_user,Dreyer2023,87,1,CSP_LDA,w_00_40,0.83125
770,cross_user,Dreyer2023,87,1,CSP_LDA,w_05_45,0.90625


In [7]:
df_list.append(
    test_accuracy(df_csp, "Dreyer2023", windows=["w_00_40", "w_05_45"], model="CSP_LDA", model_name="CSP+LDA",
                  mode="within_user"))
df_list.append(
    test_accuracy(df_csp, "Lee2019_MI", windows=["w_00_30", "w_05_35"], model="CSP_LDA", model_name="CSP+LDA",
                  mode="within_user"))
df_list.append(
    test_accuracy(df_csp, "Dreyer2023", windows=["w_00_40", "w_05_45"], model="CSP_LDA", model_name="CSP+LDA",
                  mode="cross_user"))
df_list.append(
    test_accuracy(df_csp, "Lee2019_MI", windows=["w_00_30", "w_05_35"], model="CSP_LDA", model_name="CSP+LDA",
                  mode="cross_user"))

Dreyer2023, CSP+LDA, within_user: 0.7717232334588061
Lee2019_MI, CSP+LDA, within_user: 0.0019032135127047703
Dreyer2023, CSP+LDA, cross_user: 0.0011769691482577125
Lee2019_MI, CSP+LDA, cross_user: 0.00034508486556057905


# multiple comparison correction

In [8]:
df_all = pd.concat(df_list, axis=0, ignore_index=True)
_, pvals_fdr, _, _ = multipletests(df_all["pvalue"].to_numpy(), method="fdr_bh")
df_all["pvalue_fdr"] = pvals_fdr
df_all

,dataset,model,mode,pvalue,pvalue_fdr
0,Dreyer2023,DeepConvNet,None,8.693759e-18,6.955007e-17
1,Dreyer2023,REVE,None,2.497227e-12,9.988907e-12
2,Lee2019_MI,DeepConvNet,None,1.316975e-07,3.511933e-07
3,Lee2019_MI,REVE,None,9.646089e-07,1.929218e-06
4,Dreyer2023,CSP+LDA,within_user,7.717232e-01,7.717232e-01
5,Lee2019_MI,CSP+LDA,within_user,1.903214e-03,2.175101e-03
6,Dreyer2023,CSP+LDA,cross_user,1.176969e-03,1.569292e-03
7,Lee2019_MI,CSP+LDA,cross_user,3.450849e-04,5.521358e-04
